# Notebook 04: Benchmarking Before vs After Feature Selection

## Why three AutoML tools?
- **LazyPredict:** rapid baseline ranking with minimal setup.
- **PyCaret:** end-to-end workflow abstraction and model comparison.
- **FLAML:** budget-aware optimization for practical AutoML.

We compare manual ML, LazyPredict, PyCaret, and FLAML under a consistent before/after feature-selection protocol.


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

from src.benchmark import (
    compare_before_after,
    flaml_optimize,
    lazy_predict_baseline,
    pycaret_compare,
)
from src.data_loader import load_isolet_dataset
from src.feature_selector import FeatureSelector

warnings.filterwarnings('ignore')
np.random.seed(42)

METRIC_DIR = Path('outputs/metrics')
METRIC_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
X, y, _ = load_isolet_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_train_fs, X_val_fs, y_train_fs, y_val_fs = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

selector = FeatureSelector(random_state=42)
selected_features = selector.pipeline(
    X_train_fs,
    y_train_fs,
    X_val=X_val_fs,
    y_val=y_val_fs,
    var_threshold=0.0,
    corr_threshold=0.95,
    corr_method='spearman',
    rfe_feat=180,
    rfe_step=20,
    rfe_use_cv=False,
    rfe_min_features=40,
    l1_C=1.0,
    mi_k=160,
    shap_k=120,
    verbose=False,
)

X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

print(f'Features: {X_train.shape[1]} -> {X_train_sel.shape[1]}')


## Manual model benchmark

**Models:** Logistic Regression, Random Forest, Extra Trees, SVM, KNN, XGBoost, LightGBM, CatBoost.


In [ ]:
model_specs = [
    ('LogisticRegression', LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1, multi_class='ovr')),
    ('RandomForest', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    ('ExtraTrees', ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    ('LinearSVM', LinearSVC(C=1.0, random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=7, weights='distance')),
]

manual_rows = []
for name, model in model_specs:
    comparison, _ = compare_before_after(
        model=model,
        X_train_full=X_train,
        X_test_full=X_test,
        y_train=y_train,
        y_test=y_test,
        X_train_selected=X_train_sel,
        X_test_selected=X_test_sel,
        model_name=name,
    )
    for _, row in comparison.iterrows():
        condition = 'After FS' if 'After FS' in row['model_name'] else 'Before FS'
        manual_rows.append(
            {
                'tool': 'ManualModel',
                'model_name': row['model_name'],
                'condition': condition,
                'accuracy': row['accuracy'],
                'precision': row['precision'],
                'recall': row['recall'],
                'f1': row['f1'],
                'roc_auc': row.get('roc_auc'),
                'train_time_sec': row.get('train_time_sec'),
                'inference_time_sec': row.get('inference_time_sec'),
                'train_peak_mem_mb': row.get('train_peak_mem_mb'),
                'inference_peak_mem_mb': row.get('inference_peak_mem_mb'),
                'n_features': X_train.shape[1] if condition == 'Before FS' else X_train_sel.shape[1],
            }
        )

manual_df = pd.DataFrame(manual_rows)
manual_df.head(12)


## AutoML benchmarks (LazyPredict, PyCaret, FLAML)

We use a balanced runtime profile by sampling train rows for expensive AutoML steps.


In [ ]:
sample_size = min(3500, len(X_train))
X_train_auto, _, y_train_auto, _ = train_test_split(
    X_train, y_train, train_size=sample_size, random_state=42, stratify=y_train
)
X_train_auto_sel = X_train_auto[selected_features]

automl_rows = []

# LazyPredict top-1
for condition, xtr, xte in [
    ('Before FS', X_train_auto, X_test),
    ('After FS', X_train_auto_sel, X_test_sel),
]:
    lazy_df = lazy_predict_baseline(
        xtr,
        y_train_auto,
        xte,
        y_test,
        verbose=0,
        classifiers=[RandomForestClassifier, ExtraTreesClassifier, LogisticRegression, LinearSVC, KNeighborsClassifier],
    )
    if not lazy_df.empty:
        top = lazy_df.iloc[0]
        automl_rows.append(
            {
                'tool': 'LazyPredict(top-1)',
                'model_name': str(lazy_df.index[0]),
                'condition': condition,
                'accuracy': top.get('Accuracy'),
                'precision': None,
                'recall': None,
                'f1': top.get('F1 Score'),
                'roc_auc': top.get('ROC AUC'),
                'train_time_sec': top.get('Time Taken'),
                'inference_time_sec': None,
                'train_peak_mem_mb': None,
                'inference_peak_mem_mb': None,
                'n_features': X_train.shape[1] if condition == 'Before FS' else X_train_sel.shape[1],
                'error': None,
            }
        )

# PyCaret top-1 holdout (or explicit unavailable row on Python 3.12 runtime guard)
for condition, xtr, xte in [
    ('Before FS', X_train_auto, X_test),
    ('After FS', X_train_auto_sel, X_test_sel),
]:
    train_df = xtr.copy()
    train_df['target'] = y_train_auto.values
    holdout_df = xte.copy()
    holdout_df['target'] = y_test.values

    pyc = pycaret_compare(
        train_df,
        target='target',
        fold=2,
        test_data=holdout_df,
        return_holdout=True,
        include_models=['lr', 'rf', 'et', 'knn', 'svm'],
    )
    holdout_metrics = pyc['holdout_metrics']
    leaderboard = pyc['leaderboard']
    automl_rows.append(
        {
            'tool': 'PyCaret(top-1)',
            'model_name': pyc.get('best_model_name', 'PyCaretBestModel'),
            'condition': condition,
            'accuracy': holdout_metrics.get('accuracy') if holdout_metrics else None,
            'precision': holdout_metrics.get('precision') if holdout_metrics else None,
            'recall': holdout_metrics.get('recall') if holdout_metrics else None,
            'f1': holdout_metrics.get('f1') if holdout_metrics else None,
            'roc_auc': holdout_metrics.get('roc_auc') if holdout_metrics else None,
            'train_time_sec': leaderboard.iloc[0].get('TT (Sec)') if not leaderboard.empty else None,
            'inference_time_sec': None,
            'train_peak_mem_mb': None,
            'inference_peak_mem_mb': None,
            'n_features': X_train.shape[1] if condition == 'Before FS' else X_train_sel.shape[1],
            'error': pyc.get('error'),
        }
    )

# FLAML
for condition, xtr, xte in [
    ('Before FS', X_train_auto, X_test),
    ('After FS', X_train_auto_sel, X_test_sel),
]:
    automl, result = flaml_optimize(
        xtr,
        y_train_auto,
        xte,
        y_test,
        time_budget=20,
        metric='accuracy',
        task='classification',
        estimator_list=['rf', 'extra_tree', 'sgd', 'lrl1'],
        log_training_metric=False,
    )
    automl_rows.append(
        {
            'tool': 'FLAML',
            'model_name': str(automl.best_estimator),
            'condition': condition,
            'accuracy': result['metrics'].get('accuracy'),
            'precision': result['metrics'].get('precision'),
            'recall': result['metrics'].get('recall'),
            'f1': result['metrics'].get('f1'),
            'roc_auc': result['metrics'].get('roc_auc'),
            'train_time_sec': None,
            'inference_time_sec': None,
            'train_peak_mem_mb': None,
            'inference_peak_mem_mb': None,
            'n_features': X_train.shape[1] if condition == 'Before FS' else X_train_sel.shape[1],
            'error': None,
        }
    )

automl_df = pd.DataFrame(automl_rows)
automl_df.head(12)


In [ ]:
benchmark_df = pd.concat([manual_df, automl_df], ignore_index=True)
benchmark_df.to_csv(METRIC_DIR / 'benchmark_summary.csv', index=False)

payload = {
    'dataset': 'ISOLET (OpenML did=44010)',
    'original_features': int(X_train.shape[1]),
    'selected_features': int(X_train_sel.shape[1]),
    'rows': benchmark_df.to_dict(orient='records'),
}
(METRIC_DIR / 'benchmark_summary.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')

before_after = (
    benchmark_df.groupby(['tool', 'condition'])['accuracy']
    .mean()
    .reset_index()
    .pivot(index='tool', columns='condition', values='accuracy')
    .reset_index()
)
before_after['delta_after_minus_before'] = before_after['After FS'] - before_after['Before FS']
before_after.sort_values('delta_after_minus_before', ascending=False)


## Practical interpretation

- Feature selection reduced dimensionality while preserving or improving mean accuracy in most tool families.
- Runtime and memory gains are strongest in manual model benchmarks where telemetry is fully instrumented.
- LazyPredict, PyCaret, and FLAML expose different strengths: speed, workflow abstraction, and budget-aware optimization.
